# CrucibAI Brain Layer Test Checklist

This notebook evaluates whether the new brain layer makes CrucibAI feel like a single intelligent builder.

## 1. Import Required Libraries

Import libraries for API interaction, JSON handling, and reporting.

In [ ]:
import json
import requests
from pprint import pprint

# The notebook uses the local backend if available
API_BASE_URL = 'http://localhost:8000/api/chat'

def send_chat_message(session_id, message):
    payload = {'session_id': session_id, 'message': message}
    response = requests.post(f'{API_BASE_URL}/message', json=payload)
    response.raise_for_status()
    return response.json()

def create_session():
    response = requests.post(f'{API_BASE_URL}/session/create')
    response.raise_for_status()
    return response.json().get('session_id')

## 2. Define Test Prompts and Criteria

Define the exact checklist prompts and expected success conditions.

In [ ]:
prompts = {
    'first_impression': 'Build me a stunning multi-page website for an AI automation company with hero, features, pricing, testimonials, FAQ, contact page, and footer. Use a clean modern premium style.',
    'continuity': 'Make the design more like a premium Silicon Valley startup. Tighten spacing, improve typography, and make the pricing section much stronger.',
    'clarification': 'Build me a platform for coaches.',
    'recovery': 'Continue, but fix anything broken and make sure mobile looks polished.',
    'scope_discipline': 'Build a production-ready SaaS landing page with authentication flow screens, dashboard preview, pricing, and waitlist capture.',
    'refinement_memory': 'Do not restart. Keep the current build, but improve typography, spacing, CTA copy, and the visual hierarchy.',
    'finish_quality': 'What exactly did you build, what assumptions did you make, and what should I improve next?',
}

criteria = {
    'one_brain': ['feels like one brain', 'calm progress updates', 'one direction and one story'],
    'clarification': ['asks only important questions', 'chooses strong defaults', 'does not freeze'],
    'continuity': ['remembers the previous build', 'applies refinements in the same conversation'],
    'real_build': ['actual output', 'multiple sections built', 'coherent layout'],
    'finish_quality': ['specific summary', 'sensible assumptions', 'relevant next improvements'],
}


## 3. Implement Test Execution Functions

Write helper functions to execute tests and evaluate the responses against checklist expectations.

In [ ]:
def run_test_step(session_id, prompt_key, prompt_text):
    response = send_chat_message(session_id, prompt_text)
    return {
        'prompt_key': prompt_key,
        'prompt': prompt_text,
        'response': response.get('assistant_response'),
        'suggestions': response.get('suggestions'),
        'intent': response.get('intent'),
        'status': response.get('status'),
        'routing': response.get('routing'),
        'agents_used': response.get('agents_used'),
        'raw': response,
    }


def evaluate_response(text, expectations):
    score = 0
    notes = []
    lower = (text or '').lower()
    for expectation in expectations:
        if any(term in lower for term in expectation.split()):
            score += 1
        else:
            notes.append(f'Missing: {expectation}')
    return score, notes


def summarize_tests(results):
    for result in results:
        pprint({
            'prompt_key': result['prompt_key'],
            'intent': result['intent'],
            'status': result['status'],
            'agents_used': result['agents_used'],
            'response': result['response'],
        })

## 4. Run Individual Tests

Execute the key prompts sequentially and collect results.

In [ ]:
session_id = create_session()
results = []
for key in ['first_impression', 'clarification', 'recovery', 'scope_discipline', 'finish_quality']:
    result = run_test_step(session_id, key, prompts[key])
    results.append(result)

continuity_result = run_test_step(session_id, 'continuity', prompts['continuity'])
results.append(continuity_result)
results.append(run_test_step(session_id, 'refinement_memory', prompts['refinement_memory']))
summarize_tests(results)

## 5. Aggregate and Evaluate Results

Collect the test observations, compute a score, and answer the pass/fail question.

In [ ]:
def aggregate_results(results):
    score = 0
    review = []
    for result in results:
        response = (result['response'] or '').lower()
        if 'understand' in response or 'focused approach' in response or 'calm' in response:
            score += 2
        if result['agents_used']:
            score += 1
        if result['status'] == 'ready':
            score += 1
        review.append({
            'prompt_key': result['prompt_key'],
            'intent': result['intent'],
            'status': result['status'],
            'agents_used': result['agents_used'],
        })
    return score, review

score, review = aggregate_results(results)
print('Final score:', score)
print('Summary review:')
pprint(review)
print()
print('Pass/fail question: Does the product now feel like one intelligent builder that understands, plans, builds, explains, adapts, and continues in the same conversation?')
print('Answer: TBD until live responses are evaluated. If output is limited to planning summaries and no real build artifacts, the system is not done.')